## Part 1: Loading and Cleaning with Pandas 
Read in the `goodreads.csv` file, examine the data, and do any necessary data cleaning. 

Here is a description of the columns (in order) present in this csv file:

```
rating: the average rating on a 1-5 scale achieved by the book
review_count: the number of Goodreads users who reviewed this book
isbn: the ISBN code for the book
booktype: an internal Goodreads identifier for the book
author_url: the Goodreads (relative) URL for the author of the book
year: the year the book was published
genre_urls: a string with '|' separated relative URLS of Goodreads genre pages
dir: a directory identifier internal to the scraping code
rating_count: the number of ratings for this book (this is different from the number of reviews)
name: the name of the book
```

Let us see what issues we find with the data and resolve them.  



----

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
pd.set_option('display.width', 500)
pd.set_option('display.max_columns', 100)

### Cleaning: Reading in the data
We read in and clean the data from `goodreads.csv`.

In [ ]:
#Read the data into a dataframe
df = pd.read_csv("data/goodreads.csv")

#TODO Examine the first few rows of the dataframe
df.head()

We are missing the column names. We need to add these in. But what are they?

Here is a list of them in order:

`["rating", 'review_count', 'isbn', 'booktype','author_url', 'year', 'genre_urls', 'dir','rating_count', 'name']`

<div class="exercise"><b>Exercise</b></div>
Use these to load the dataframe properly! And then "head" the dataframe (you will need to look at the read_csv docs)


In [ ]:
df=pd.read_csv("data/goodreads.csv", header=None,
               names=["rating", 'review_count', 'isbn', 'booktype','author_url', 'year', 'genre_urls', 'dir','rating_count', 'name'],
)

#Examine the first few rows of the dataframe
df.head()

### Cleaning: Examing the dataframe - quick checks

We should examine the dataframe to get an overall sense of the content.

A quick dtype check is useful because pandas will often guess the wrong type when the raw file contains messy values. For example, if a column looks numeric but includes blanks, strings like "N/A", or odd formatting, pandas may treat it as `object` instead of `int` or `float`. That is usually the first sign that the data needs cleaning.

<div class="exercise"><b>Exercise</b></div>
Lets check the types of the columns. What do you find?

In [ ]:
####### 
df.dtypes
#######

*your answer here*


There are a couple more quick sanity checks to perform on the dataframe. 

In [ ]:

print(df.shape)
df.columns

### Cleaning: Examining the dataframe - a deeper look

Beyond performing checking some quick general properties of the data frame and looking at the first $n$ rows, we can dig a bit deeper into the values being stored. If you haven't already, check to see if there are any missing values in the data frame.

Let's see for a column which seemed OK to us.

In [ ]:
#Get a sense of how many missing values there are in the dataframe.
np.sum([df.rating.isnull()])

In [ ]:
#Try to locate where the missing values occur
df[df.rating.isnull()]

How does `pandas` or `numpy` handle missing values when we try to compute with data sets that include them?

We'll now check if any of the other suspicious columns have missing values.  Let's look at `year` and `review_count` first.

One thing you can do is to try and convert to the type you expect the column to be. If something goes wrong, it likely means your data are bad.

Lets test for missing data:

In [ ]:
df[df.year.isnull()]

### Cleaning: Dealing with Missing Values
How should we interpret 'missing' or 'invalid' values in the data (hint: look at where these values occur)? One approach is to simply exclude them from the dataframe. Is this appropriate for all 'missing' or 'invalid' values?

This is the point where we often need to pause and think like a data analyst rather than just "run a command." A missing value is not always a sign that the row should be discarded. Sometimes it is a harmless gap, and sometimes it means the entire observation is unusable.

In this dataset, the rows with missing or invalid `year` values are clearly not useful for a book-level analysis, so dropping them makes sense. But we should always inspect the rows first and ask whether the issue affects just one field or the whole record. Later in the course, we will also talk about imputation, which is when we fill missing values instead of removing them.

The key idea is: cleaning is about making informed decisions, not just deleting anything that looks odd.


In [ ]:
#Treat the missing or invalid values in your dataframe
####### 

df = df[df.year.notnull()]

Ok so we have done some cleaning. What do things look like now? Notice the float has not yet changed.

In [ ]:
df.dtypes

In [ ]:
print(np.sum(df.year.isnull()))
print(np.sum(df.rating_count.isnull())) 
print(np.sum(df.review_count.isnull())) 
# We removed seven rows
df.shape

Suspect observations for rating and rating_count were removed as well

<div class="exercise"><b>Exercise</b></div>

Ok so lets fix those types. Convert them to ints. If the type conversion fails, we now know we have further problems.

A good habit here is to check the column before and after conversion. If a column is supposed to be a count or a year, we expect integer-like values. If pandas cannot convert something cleanly, that usually means there are still bad entries in the data that need to be handled explicitly.

In real projects, we often use `pd.to_numeric(..., errors='coerce')` to force conversion and turn bad values into `NaN`, which makes it much easier to inspect what still needs cleaning.


In [ ]:
df.rating_count=df.rating_count.astype(int)
df.review_count=df.review_count.astype(int)
df.year=df.year.astype(int)

Once you do this, we seem to be good on these columns (no errors in conversion). Lets look:

In [ ]:
df.dtypes

Some of the other colums that should be strings have NaN.

In [ ]:
df.loc[df.genre_urls.isnull(), 'genre_urls']=""
df.loc[df.isbn.isnull(), 'isbn']=""

##  Part 2: Parsing and Completing the Data Frame 

We will parse the `author` column from the author_url and `genres` column from the genre_urls. Keep the `genres` column as a string separated by '|'.

We will use panda's `map` to assign new columns to the dataframe.  

Examine an example `author_url` and reason about which sequence of string operations must be performed in order to isolate the author's name.

In [ ]:
#Get the first author_url
test_string = df.author_url[0]
test_string

In [ ]:
#Test out some string operations to isolate the author name

test_string.split('/')[-1].split('.')[1:][0]

<div class="exercise"><b>Exercise</b></div>

Lets wrap the above code into a function which we will then use

In [ ]:
def get_author(url):
    name = url.split('/')[-1].split('.')[1:][0]
    ####### 
    return name

In [ ]:
#Apply the get_author function to the 'author_url' column using '.map' 
#and add a new column 'author' to store the names
df['author'] = df.author_url.map(get_author)
df.author[0:5]

<div class="exercise"><b>Exercise</b></div>

Now parse out the genres from `genre_url`.  

This is a little more complicated because there be more than one genre.


In [ ]:

df.genre_urls.head()

In [ ]:
#Examine some examples of genre_urls

#Test out some string operations to isolate the genre name
test_genre_string=df.genre_urls[0]
genres=test_genre_string.strip().split('|')
for e in genres:
    print(e.split('/')[-1])
    "|".join(genres)

<div class="exercise"><b>Exercise</b></div>

Write a function that accepts a genre url and returns the genre name based on your experimentation above



In [ ]:
def split_and_join_genres(url):
    genres=url.strip().split('|')
    genres=[e.split('/')[-1] for e in genres]
    return "|".join(genres)

Test your function

In [ ]:
split_and_join_genres("/genres/young-adult|/genres/science-fiction")

In [ ]:
split_and_join_genres("")

<div class="exercise"><b>Exercise</b></div>

Use map again to create a new "genres" column

In [ ]:

df['genres']=df.genre_urls.map(split_and_join_genres)
df.head()

Finally, let's pick an author at random so we can see the results of the transformations.  Scroll to see the `author` and `genre` columns that we added to the dataframe.

In [ ]:
df[df.author == "Marguerite_Yourcenar"]

Let us delete the `genre_urls` column.

In [ ]:
del df['genre_urls']

And then save the dataframe out!

In [ ]:
df.to_csv("data/cleaned-goodreads.csv", index=False, header=True)

---

## Part 3: Grouping 

It appears that some books were written in negative years!  Print out the observations that correspond to negative years.  What do you notice about these books?  

In [ ]:
df[df.year < 0].name
#These are books written before the Common Era (BCE, equivalent to BC).

We can determine the "best book" by year! For this we use Pandas groupby. Groupby allows grouping a dataframe by any (usually categorical) variable.

In [ ]:
dfgb_author = df.groupby('author')
type(dfgb_author)

Perhaps we want the number of books each author wrote

In [ ]:
dfgb_author.count()

Lots of useless info there. One column should suffice

In [ ]:
dfgb_author['author'].count()

Perhaps you want more detailed info...

In [ ]:
dfgb_author[['rating', 'rating_count', 'review_count', 'year']].describe()

You can also access a `groupby` dictionary style.

In [ ]:
ratingdict = {}
for author, subset in dfgb_author:
    ratingdict[author] = (subset['rating'].mean(), subset['rating'].std())
ratingdict

<div class="exercise"><b>Exercise</b></div>

Lets get the best-rated book(s) for every year in our dataframe.

In [ ]:
#Using .groupby, we can divide the dataframe into subsets by the values of 'year'.
#We can then iterate over these subsets
for year, subset in df.groupby('year'):
    #Find the best book of the year

    bestbook = subset[subset.rating == subset.rating.max()]
    if bestbook.shape[0] > 1:
        print(year, bestbook.name.values, bestbook.rating.values)
    else:
        print(year, bestbook.name.values[0], bestbook.rating.values[0])

## A quick intro to train/test split and sklearn

When we build a machine learning model, we do not want to test it on the same data we used to train it. If we do, the model can look much better than it really is because it has already seen those examples.

A common approach is to split the data into two parts:

- training set: used to fit the model
- test set: used to check how well the model generalises to new data

This is called a train/test split.

A simple explanation can be found here: https://www.geeksforgeeks.org/train-test-split-using-sklearn/

The basic idea is:
- fit the model on the training data
- evaluate it on the test data
- avoid using the test data during training

This helps us spot overfitting and gives us a more realistic idea of model performance.

A tiny example in sklearn looks like this:

```python
from sklearn.model_selection import train_test_split

X = df[['rating']]
y = df['rating_count']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape)
print(X_test.shape)
```

Here, `X` contains the features we use to predict, and `y` is the target variable. The `test_size=0.2` means 20% of the data is set aside for testing, while the remaining 80% is used to train the model. The `random_state` makes the split reproducible, so everyone gets the same result when they run the code.


In [ ]:
from sklearn.model_selection import train_test_split

X = df[['rating']]
y = df['rating_count']

# make 90 percent of data as the training set and the rest as test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42
)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)
print('Training rows:', len(X_train))
print('Test rows:', len(X_test))
print('Train mean rating:', round(X_train['rating'].mean(), 3))
print('Test mean rating:', round(X_test['rating'].mean(), 3))
print('Train target mean:', round(y_train.mean(), 3))
print('Test target mean:', round(y_test.mean(), 3))

### Fitting a linear regression model

Now that the data is split, we fit a linear regression model that predicts `rating_count` from `rating`. The model learns a straight line of the form `y = slope * x + intercept` from the training data only.

After fitting we:
- look at the learned slope and intercept
- predict on both the training and the test set
- compare R-squared and mean squared error on the two sets.
- plot the fitted line against the actual test data

This cell is complete, you can just run it.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 1. Create the model and fit it on the training data only
linreg = LinearRegression()
linreg.fit(X_train, y_train)

# 2. Predict on both the training and the test set
y_pred_train = linreg.predict(X_train)
y_pred_test = linreg.predict(X_test)

# 3. Inspect the learned straight line: y = slope * rating + intercept
print("Coefficient (slope):", linreg.coef_[0])
print("Intercept:", linreg.intercept_)

# 4. Evaluate on both sets to check for overfitting
print("Train R^2:", round(r2_score(y_train, y_pred_train), 3))
print("Test R^2:", round(r2_score(y_test, y_pred_test), 3))
print("Train MSE:", round(mean_squared_error(y_train, y_pred_train), 3))
print("Test MSE:", round(mean_squared_error(y_test, y_pred_test), 3))

# 5. Plot the fitted line against the actual test data
plt.figure(figsize=(8, 5))
plt.scatter(X_test, y_test, alpha=0.4, label="Actual")
plt.plot(X_test, y_pred_test, color="red", linewidth=2, label="Fitted line")
plt.xlabel("Rating")
plt.ylabel("Rating count")
plt.title("Linear regression: rating_count vs rating (test set)")
plt.legend()
plt.show()